# 第2课：向量嵌入与索引构建

## 本节目标

理解如何将文本转化为计算机可以比较的数学表示（向量），并构建高效的检索索引。你将学会：

1. **文本向量化**：使用 Sentence-Transformer 将文本转为语义向量
2. **向量相似度**：理解余弦相似度的含义
3. **向量存储**：使用 ChromaDB 和 FAISS 存储和检索向量
4. **嵌入模型对比**：不同模型对检索效果的影响

> 核心直觉：语义相近的文本，在向量空间中距离也近。"猫"和"小猫"的向量距离远小于"猫"和"汽车"。

## 环境准备

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from config import config
from src.document_loader import DocumentLoader
from src.chunker import Chunker
from src.embedder import Embedder
from src.vector_store import VectorStore

print("模块加载成功!")

## 1. 准备数据

先加载文档并分块（复用第1课的知识）。

In [ ]:
# 加载文档
loader = DocumentLoader()
docs = loader.run()

# 分块
chunker = Chunker(strategy="recursive_char", chunk_size=512, chunk_overlap=50)
chunks = chunker.chunk_documents(docs)
texts = [c["content"] for c in chunks]

print(f"准备了 {len(texts)} 个文本块")

## 2. 文本向量化

`sentence-transformers` 将每个文本块映射到一个固定长度的浮点数向量。这个向量的每一维没有明确的人类可读含义——它是在大量数据上训练出来的语义空间坐标。

### 支持的模型
- **all-MiniLM-L6-v2**：轻量英文模型（384维，速度快）
- **BAAI/bge-small-zh-v1.5**：BGE 中文模型（512维，中英文兼顾）
- **moka-ai/m3e-base**：M3E 中文模型（768维，中文效果好）

In [ ]:
# 初始化嵌入模型（首次运行会下载模型权重 ~100-400MB）
embedder = Embedder("BAAI/bge-small-zh-v1.5")
print(f"模型: {embedder.model_name}")
print(f"向量维度: {embedder.dim}")

# 嵌入所有文本块
embeddings = embedder.embed(texts, show_progress=True)
print(f"\n嵌入结果: {embeddings.shape}")  # (N, dim)

## 3. 直观感受向量相似度

两个文本的语义越接近，它们的向量夹角越小（余弦相似度越高）。

In [ ]:
# 编码几个句子
sentences = [
    "机器学习是人工智能的一个分支",
    "深度学习使用神经网络进行学习",
    "今天天气真不错适合出去玩",
]
sent_embs = embedder.embed(sentences, show_progress=False)

# 计算成对相似度
for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if i < j:
            sim = np.dot(sent_embs[i], sent_embs[j])  # 向量已归一化，点积 = 余弦相似度
            print(f"sim('{s1[:20]}...', '{s2[:20]}...') = {sim:.4f}")

## 4. 构建向量索引：ChromaDB

ChromaDB 是一个开源的向量数据库，提供持久化存储、元数据管理和高效的近似最近邻（ANN）搜索。

In [ ]:
# 初始化 ChromaDB 向量存储
chroma_store = VectorStore(store_type="chroma", collection_name="tutorial_demo")

# 将分块和嵌入一起存入
chroma_store.add(chunks, embeddings)
print(f"存储了 {chroma_store.count()} 个向量")

## 5. 检索测试

用查询向量去搜索最相似的文本块。

In [ ]:
# 向量化查询
query = "What is gradient descent?"
query_emb = embedder.embed_query(query)

# 搜索 Top-3
results = chroma_store.query(query_emb, top_k=3)

print(f"查询: {query}\n")
for i, r in enumerate(results, 1):
    print(f"--- Top {i} (相似度: {r['score']:.4f}) ---")
    print(f"来源: {r['metadata']['source']}")
    print(f"内容: {r['content'][:200]}...")
    print()

## 6. FAISS 对比

FAISS 是 Meta 开源的高性能向量检索库，纯内存运行，速度极快但在进程结束后不持久化。

In [ ]:
# 切换为 FAISS
try:
    faiss_store = VectorStore(store_type="faiss", collection_name="tutorial_demo")
    faiss_store.add(chunks, embeddings)
    print(f"FAISS 存储了 {faiss_store.count()} 个向量")

    # 用相同查询检索
    faiss_results = faiss_store.query(query_emb, top_k=3)

    print(f"\n查询: {query}\n")
    for i, r in enumerate(faiss_results, 1):
        print(f"--- Top {i} (相似度: {r['score']:.4f}) ---")
        print(f"来源: {r['metadata']['source']}")
        print()
except ImportError:
    print("FAISS 未安装。安装方法: pip install faiss-cpu")

## 7. ChromaDB vs FAISS 选型建议

| 特性 | ChromaDB | FAISS |
|------|----------|-------|
| 持久化 | 内置 SQLite | 需手动序列化 |
| 元数据过滤 | 原生支持 | 需自行实现 |
| 检索速度 | 中等 (HNSW) | 极快 (GPU可选) |
| 部署复杂度 | 低（嵌入式） | 低 |
| 适用场景 | 原型和小规模应用 | 大规模/高性能场景 |

> 本教程默认使用 ChromaDB，因为它开箱即用，不需要额外处理持久化。

## 本节小结

- 文本向量化是 RAG 的核心——它将语义比较转化为数学计算
- 嵌入模型的选择影响语义理解的准确性
- ChromaDB 适合原型开发，FAISS 适合追求极致性能
- 向量归一化后，点积直接等于余弦相似度

**下一步**：在 03 号笔记本中，我们将学习如何使用 BM25 + 向量搜索进行混合检索，并接入 LLM 生成答案。